# 03 Statistical Analysis

Project: Predicting Thyroid Dysfunction Using Machine Learning

Author: Namitha Nair

This notebook is part of a cleaned research workflow for the NHANES thyroid-metabolic analysis.


## Purpose

This notebook tests whether metabolic variables differ across TSH groups using non-parametric statistical methods. Kruskal-Wallis tests are used for overall group differences, followed by Dunn's post-hoc tests with Bonferroni correction.


In [ ]:
from pathlib import Path

# Project paths. These work if the notebooks are run from the repository root.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIGURES_DIR = Path("figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Install only if needed. In Colab, uncomment the next line if scikit-posthocs is not installed.
# !pip install scikit-posthocs


In [ ]:
import pandas as pd
from scipy.stats import kruskal
import scikit_posthocs as sp

DATA_PATH = PROCESSED_DIR / "NHANES_Analysis_Dataset.csv"
analysis_df = pd.read_csv(DATA_PATH)

print(analysis_df.shape)
analysis_df[["TSH_Group"]].value_counts()


## Kruskal-Wallis tests

The Kruskal-Wallis test checks whether at least one TSH group differs from the others for each metabolic variable.


In [ ]:
variables = [
    "Fasting_Glucose_mg_dL",
    "HbA1c_Percent",
    "BMI",
    "Fasting_Insulin_uU_mL"
]

kruskal_results = []

for var in variables:
    normal = analysis_df[analysis_df["TSH_Group"] == "Normal"][var].dropna()
    high = analysis_df[analysis_df["TSH_Group"] == "High"][var].dropna()
    low = analysis_df[analysis_df["TSH_Group"] == "Low"][var].dropna()

    stat, p = kruskal(normal, high, low)
    kruskal_results.append({
        "Variable": var,
        "Kruskal_Wallis_Statistic": stat,
        "P_Value": p,
        "Significant_0.05": p < 0.05
    })

kruskal_results_df = pd.DataFrame(kruskal_results)
kruskal_results_df


## Dunn's post-hoc tests

Dunn's test identifies which pairs of TSH groups differ after a significant Kruskal-Wallis result.


In [ ]:
dunn_results = {}

for var in variables:
    result = sp.posthoc_dunn(
        analysis_df,
        val_col=var,
        group_col="TSH_Group",
        p_adjust="bonferroni"
    )
    dunn_results[var] = result
    print("
" + "="*70)
    print(var)
    display(result)


## Save statistical result tables


In [ ]:
kruskal_results_df.to_csv(PROCESSED_DIR / "kruskal_results.csv", index=False)

for var, result in dunn_results.items():
    safe_name = var.lower().replace("/", "_").replace(" ", "_")
    result.to_csv(PROCESSED_DIR / f"dunn_{safe_name}.csv")

print("Saved statistical results to data/processed/")


## Notes for manuscript

Initial result summary:

- Fasting glucose, HbA1c, BMI, and fasting insulin differed significantly across TSH groups in Kruskal-Wallis testing.
- Dunn's post-hoc tests showed the clearest differences for High TSH vs Normal TSH in fasting glucose and HbA1c.
- BMI showed a near-significant High vs Normal comparison after Bonferroni correction in the preliminary analysis.
- Insulin differences appeared strongly influenced by the Low TSH group, which had a smaller sample size.
